# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hassanqureshi46278-art/flyrank-Internship-ML/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row in fact_content_daily_performance = one content page's performance on one calendar day. My lane rolls these up to page-level over a mid-panel month, month=2026-03, rather than the _sample table (June 2026), since the _sample is the sealed final month and using it to build label logic would mean training on the same window I'd eventually need as an honest outcome check.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Feature	daily clicks, impressions, avg position, CTR, days since last content update — all knowable before the decision date
Label/proxy	a forward-looking outcome (e.g., traffic change in the following window) if one exists at this grain, or my w01/w02 staleness+demand+underperformance proxy computed from the same month
Context	content/page hash key, client hash key, month/date (used to group and filter, never fed to the model as a "feature" since it's an ID, not a signal)
Excluded	fact_content_query_90d (query-level hashed keywords) — dropped for this lane to avoid an unnecessary join and any risk of pulling in forward-looking query data; anything from dim_clients beyond gsc_data_start/ga4_data_start — excluded per the dataset's anonymization terms

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [3]:
import duckdb
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"

# 0) Inspect the real schema first — don't guess column names
print(con.sql(f"DESCRIBE SELECT * FROM read_parquet('{REL}/fact_content_daily_performance/**/*.parquet') LIMIT 1"))
features_df = con.sql(f"""
    SELECT
        content_key,
        AVG(clicks)      AS avg_clicks_28d,       -- knowable: trailing daily clicks, no future data used
        AVG(impressions)  AS avg_impressions_28d, -- knowable: trailing daily impressions
        AVG(position)     AS avg_position_28d,    -- knowable: trailing rank, observed daily
        AVG(clicks) / NULLIF(AVG(impressions), 0) AS ctr_28d, -- knowable: derived from the two features above
        DATEDIFF('day', MAX(last_updated_date), DATE '{MONTH}-01') AS days_since_update -- knowable: update timestamp precedes decision date
    FROM read_parquet('{REL}/fact_content_daily_performance/month={MONTH}/*.parquet')
    GROUP BY content_key
""").df()
features_df.head(10)
# Deliberate leak: pull in something only knowable AFTER the decision (e.g. next month's clicks)
leaky_df = features_df.copy()
leaky_df["next_month_clicks"] = con.sql(f"""
    SELECT content_key, SUM(clicks) AS next_month_clicks
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-04/*.parquet')
    GROUP BY content_key
""").df().set_index("content_key").reindex(leaky_df["content_key"]).values

# quick score with the leaked column in the feature set — watch it jump toward ~1.0
# ... fit a quick baseline model with and without next_month_clicks here ...

# then remove it and keep the honest, lower number
leaky_df = leaky_df.drop(columns=["next_month_clicks"])
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

# Build a simple proxy target to score against — e.g. this month's avg_clicks_28d
# (swap for your actual w01/w02 proxy target if you already built one)
target_col = "avg_clicks_28d"

def quick_score(df, feature_cols, target_col):
    data = df.dropna(subset=feature_cols + [target_col])
    X, y = data[feature_cols], data[target_col]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    model = LinearRegression().fit(X_train, y_train)
    return r2_score(y_test, model.predict(X_test))

honest_features = ["avg_impressions_28d", "avg_position_28d", "ctr_28d", "days_since_update"]
leaky_features = honest_features + ["next_month_clicks"]

print(f"Honest R²  (no leak): {quick_score(leaky_df, honest_features, target_col):.3f}")
print(f"Leaky R²  (with leak): {quick_score(leaky_df, leaky_features, target_col):.3f}")

# then remove it and keep the honest, lower number
leaky_df = leaky_df.drop(columns=["next_month_clicks"])
print(f"\nFinal honest features kept: {honest_features}")

SecretNotFoundError: Secret HF_TOKEN does not exist.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This slice can never tell me why a page's performance changed — only that it did. The panel is unbalanced (dim_clients.gsc_data_start/ga4_data_start differ per client), so early rows for newer clients may be GSC-only with no GA4 signal, which could quietly bias any feature that assumes both sources exist. And because fact_content_query_90d uses salted, per-content aggregate shares for the rare tail (<10 impressions), any query-level rollup will undercount true long-tail volume rather than reporting zero, which is an easy thing to misread as "no demand."

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.